# ETL pipeline for DOE

In [1]:
import pandas as pd
import json

In [2]:
def extract_tabular_data(file_path: str):
    """Extract data from a tabular file_format, with pandas."""
    if file_path.endswith(".csv"):
        return pd.read_csv(file_path)

    elif file_path.endswith(".parquet"):
        return pd.read_parquet(file_path)

    else: 
        raise Exception("Warning: Invalid file extension. Please try with .csv or .parquet!")

In [3]:
def extract_json_data(file_path):
    """Extract and flatten data from a JSON file."""
    with open(file_path, "r") as json_file:
        raw_data = json.load(json_file)

    return pd.json_normalize(raw_data)

In [4]:
def transform_electricity_sales_data(raw_data: pd.DataFrame):
    
    raw_data.dropna(subset=["price"], inplace=True)
  
    cleaned_df = raw_data.loc[raw_data["sectorName"].isin(["residential", "transportation"]), :]
  
    cleaned_df["month"]= cleaned_df["period"].str[4:0]
    cleaned_df["year"]= cleaned_df["period"].str[0:2]
    
    cleaned_df = cleaned_df.loc[:, ["year", "month", "stateid", "price", "price-units"]]

    return cleaned_df
    

In [5]:
def load(dataframe: pd.DataFrame, file_path: str):
     if file_path.endswith(".csv"):
         dataframe.to_csv(file_path)

     elif file_path.endswith(".parquet"):
           dataframe.to_parquet(file_path)

     else: 
        raise Exception("Warning: {file_path} is not a valid file type. Please try again!")

In [6]:

raw_electricity_capability_df = extract_json_data("electricity_capability_nested.json")
raw_electricity_sales_df = extract_tabular_data("electricity_sales.csv")

# Ensure the transform function returns a DataFrame
cleaned_electricity_sales_df = transform_electricity_sales_data(raw_electricity_sales_df)

load(raw_electricity_capability_df, "loaded__electricity_capability.parquet")
load(cleaned_electricity_sales_df, "loaded__electricity_sales.csv")

print(cleaned_electricity_sales_df)
print(raw_electricity_capability_df)

     year month stateid  price              price-units
0      20            HI   0.00  cents per kilowatt-hour
5      20            ID  11.53  cents per kilowatt-hour
6      20            ID   0.00  cents per kilowatt-hour
11     20            IL  14.84  cents per kilowatt-hour
12     20            IL   7.29  cents per kilowatt-hour
...   ...   ...     ...    ...                      ...
4821   20            IA   0.00  cents per kilowatt-hour
4826   20            KS  13.48  cents per kilowatt-hour
4827   20            KS   0.00  cents per kilowatt-hour
4832   20            KY  12.51  cents per kilowatt-hour
4833   20            KY   0.00  cents per kilowatt-hour

[1612 rows x 5 columns]
     period stateId  ... energySource.capability energySource.capabilityUnits
0      2023      WI  ...                  1442.6                    megawatts
1      2023      WI  ...                  1442.6                    megawatts
2      2023      WI  ...                   636.5                    m